hive-video-examples.ipynb

Peter Dresslar

2026/09/07

Demonstrations of the tools available from the repository, https://github.com/Collective-Logic-Lab/honeybee-hive-video

In [1]:
# Options:
#
# We have a number of ways we can use the hive-video tools, but the easiest is to install the module from PyPi.
# 
# Here, we call the install with the optional add-on `resequence`, which is a larger, more complicated project than the other tools
# Your usage may just need `hive-video`, in which case you can leave out the bracketed extra for resequence.

# %pip install "hive-video[resequence]"  # only need to install once in an environment
import hive_video

print(hive_video.__version__)   # should print 0.1.1 as of this notebook



0.1.1


One of the tools you can use from Honeybee Hive Video is the download tool. This will allow you to programmatically obtain and store videos from the Edmond repository (https://edmond.mpg.de/dataset.xhtml?persistentId=doi:10.17617/3.LLWRWR).

A couple of things about the video files:

1. They are very large. Some of the videos are 125GB+ (!!) ... Most are around 25-35GB. That is for one mp4 file.
2. They are organized with a particular order:
- Day (sequential day of trial)
- Side
- Panel

So for instance the filename:
	
`start01__20190606_190340_side0_bottom.mp4`

...means day 1 (couning from day zero); side 0, and bottom panel. What `day` means is obvious, but you may be interested to know more about `side` and `panel`: an illustrative guide appears here: https://edmond.mpg.de/file.xhtml?fileId=238043&version=1.0. As a team we have generally been pursuing understanding of the *top* panel videos, as they have more space for flight. We have worked with both sides 0 and 1 in the past.

So, let's say we would like to acquire the video from day 56, side 0, and top panel: `start56__20190809_174417_side0_top.mp4`. Our download tool allows us to shorthand the video in case we do not have the exact date and time handy. **This video is about 25GB in size.**

In [2]:
from pathlib import Path
from hive_video.download import download_video

my_dir = Path("data/raw/day56/")

# The following line is intentionally commented out! Uncomment to download. Downloads can take many minutes or hours.
# source = download_video(day=56, side=0, panel="top", target=my_dir)


This large download is likely to take a very long time. So long, in fact, that we might not want to run it at all from the `hive-video` api within a python script. Our alternative is to use the `hive-video` CLI, which--given the nature of the website and the filenames involved, can be very convenient. Running from the cli would look like:

```bash
uv tool install hive-video    # Or you could use pip
hive-video --version          
# prints: hive-video 0.1.1

hive-video download --start 56 --side 0 --panel top --target data/raw/day56/ --timeout 120
```
Note that in the command line call we have added a timeout value. There are several options like this one: view them with

```bash
hive-video download --help
```

If we can execute downloads using the CLI, why have the option to use Python and the API? The answer is that it is particularly useful to use the python API for pipelining scripts on a job-controlled, high-performance computing environment like ASU's Sol.

---
Moving to our next, most versatile function, we have our fragment function. Since the videos are so large and so processor intensive to download or even view, it can be far more useful to manually locate and cut out "snippets" (fragments) of the videos one can then continue analyzing. Our `fragment` tool does this.

To see `fragment` in action, we have a small, five-second video included with this repository that is sized well below the Github file size limit. It is located at `data/raw/start04_sample_5s.mp4`.

Let's say that to start, we would like to cut the video into one-second chunks. Here's how we can do that using the `hive-video` API:

In [6]:
from hive_video.fragment import create_fragment

fragment_input = Path("data/raw/start04_sample_5s.mp4")  # Must run from project root!
output_dir = Path("data/artifacts/start04_samples_seconds")  # Must run from project root!
output_dir.mkdir(parents=True, exist_ok=True)        # Ensures the folder exists

dur = 1  # duration of 1 second each.

for i in range(5):
    out_path = output_dir / f"clip_{i:02d}.mp4"
    clip = create_fragment(
        str(fragment_input),
        str(out_path),
        start=i * dur,
        duration=dur,
        unit="seconds",
    )
    print(clip)

🔽 Downloading from https://www.osxexperts.net/ffmpeg80arm.zip
✅ Download complete!
🔽 Downloading from https://www.osxexperts.net/ffprobe80arm.zip
✅ Download complete!


/Users/peterdresslar/Workspace/honey-bee-behavior/data/artifacts/start04_samples/clip_00.mp4
/Users/peterdresslar/Workspace/honey-bee-behavior/data/artifacts/start04_samples/clip_01.mp4
/Users/peterdresslar/Workspace/honey-bee-behavior/data/artifacts/start04_samples/clip_02.mp4
/Users/peterdresslar/Workspace/honey-bee-behavior/data/artifacts/start04_samples/clip_03.mp4
/Users/peterdresslar/Workspace/honey-bee-behavior/data/artifacts/start04_samples/clip_04.mp4


We might also decide that we want to work with precise frame numbers to take a video fragment. Notice that in the sample video we do have frame numbers at the top that we can work with. Instead of taking 1 second slices, let's take 25 frame slices, which should actually be equivalent as this video is encoded at 25fps.

In [ ]:
# fragment_input = Path("data/raw/start04_sample_5s.mp4")  # Already set.
output_dir = Path("data/artifacts/start04_samples_frames")  # Must run from project root!
output_dir.mkdir(parents=True, exist_ok=True)        # Ensures the folder exists

dur = 25  # duration of 1 second each.

for i in range(5):
    out_path = output_dir / f"clip_{i:02d}.mp4"
    clip = create_fragment(
        str(sample_vid),
        str(out_path),
        start=i * dur,
        duration=dur,
        unit="frames",
    )
    print(clip)